# Analyse du marché immobilier - Loire-Atlantique (DVF)
## Contexte et objectifs
L'objectif est d'étudier l'évolution du marché immobilier en Loire Atlantique entre 2021 et 2025.
## Import des données

In [ ]:
import pandas as pd

dept = "44"
annees = [2021, 2022, 2023, 2024, 2025]
dfs = []

for annee in annees:
    url = f"https://files.data.gouv.fr/geo-dvf/latest/csv/{annee}/departements/{dept}.csv.gz"
    try:
        df_annee = pd.read_csv(url, compression="gzip", low_memory=False)
        df_annee["annee_mutation_fichier"] = annee  
        dfs.append(df_annee)
        print(f"{annee} : {df_annee.shape[0]} lignes chargées")
    except Exception as e:
        print(f"Erreur pour {annee} : {e}")


On fusionne ensuite toutes les données en un seul Dataframe

In [ ]:
df = pd.concat(dfs, ignore_index=True)
print("\nDataset final :")
print(df.shape)

In [5]:
df.head()


,id_mutation,date_mutation,numero_disposition,nature_mutation,valeur_fonciere,adresse_numero,adresse_suffixe,adresse_nom_voie,adresse_code_voie,code_postal,...,surface_reelle_bati,nombre_pieces_principales,code_nature_culture,nature_culture,code_nature_culture_speciale,nature_culture_speciale,surface_terrain,longitude,latitude,annee_mutation_fichier
0,2021-744798,2021-01-05,1,Vente,130000.0,10.0,NaN,RUE RICHER,7308,44100.0,...,45.0,1.0,NaN,NaN,NaN,NaN,NaN,-1.576574,47.212731,2021
1,2021-744798,2021-01-05,1,Vente,130000.0,10.0,NaN,RUE RICHER,7308,44100.0,...,NaN,0.0,NaN,NaN,NaN,NaN,NaN,-1.576574,47.212731,2021
2,2021-744799,2021-01-04,1,Vente,700000.0,3.0,NaN,RUE DES MARTYRS,5444,44100.0,...,134.0,5.0,S,sols,NaN,NaN,195.0,-1.575733,47.216779,2021
3,2021-744800,2021-01-05,1,Vente,354240.0,48.0,B,RUE DE COULMIERS,2244,44000.0,...,74.0,3.0,NaN,NaN,NaN,NaN,NaN,-1.536938,47.222669,2021
4,2021-744801,2021-01-11,1,Vente,30000.0,21.0,NaN,PAS PETITES SOEURS DES PAUVRES,6505,44000.0,...,NaN,0.0,NaN,NaN,NaN,NaN,NaN,-1.561705,47.225324,2021


## Exploration du Dataset:
On regarde la répartition de nos données selon le type de bien:

In [10]:
df["type_local"].value_counts(dropna=False)

type_local
NaN                                         187493
Dépendance                                  108611
Maison                                       80957
Appartement                                  50478
Local industriel. commercial ou assimilé     11067
Name: count, dtype: int64

On explore nos colonnes et leur signification

In [12]:
df.columns.tolist()

['id_mutation',
 'date_mutation',
 'numero_disposition',
 'nature_mutation',
 'valeur_fonciere',
 'adresse_numero',
 'adresse_suffixe',
 'adresse_nom_voie',
 'adresse_code_voie',
 'code_postal',
 'code_commune',
 'nom_commune',
 'code_departement',
 'ancien_code_commune',
 'ancien_nom_commune',
 'id_parcelle',
 'ancien_id_parcelle',
 'numero_volume',
 'lot1_numero',
 'lot1_surface_carrez',
 'lot2_numero',
 'lot2_surface_carrez',
 'lot3_numero',
 'lot3_surface_carrez',
 'lot4_numero',
 'lot4_surface_carrez',
 'lot5_numero',
 'lot5_surface_carrez',
 'nombre_lots',
 'code_type_local',
 'type_local',
 'surface_reelle_bati',
 'nombre_pieces_principales',
 'code_nature_culture',
 'nature_culture',
 'code_nature_culture_speciale',
 'nature_culture_speciale',
 'surface_terrain',
 'longitude',
 'latitude',
 'annee_mutation_fichier']

La colonne "code_nature_culture" expose la nature des terrains achetés: S correspond par exemple à 
Sols (terrain bâti, sous une construction),T à Terres(terres agricoles labourables), J correspond à Jardins etc. En observant la répartition de nos données on constate une grande majrité de terrains de type S.





In [ ]:
df["code_nature_culture"].value_counts(dropna=False)

## Nettoyage
Dans le cadre de notre étude nous allons nous concentrer sur les logements. Nous pourrons réaliser une seconde étude sur les terrains plus tard.
Nous conservons les locaux de type "Maison" et "Appartement"


In [26]:
df_residentiel = df[df["type_local"].isin(["Maison", "Appartement"])].copy()

Nous traitons ensuite les valeurs nulles:

In [27]:
df_residentiel.isna().sum()

id_mutation                          0
date_mutation                        0
numero_disposition                   0
nature_mutation                      0
valeur_fonciere                    333
adresse_numero                     336
adresse_suffixe                 123009
adresse_nom_voie                     0
adresse_code_voie                    0
code_postal                          0
code_commune                         0
nom_commune                          0
code_departement                     0
ancien_code_commune             131435
ancien_nom_commune              131435
id_parcelle                          0
ancien_id_parcelle              131435
numero_volume                   131435
lot1_numero                      83899
lot1_surface_carrez             104582
lot2_numero                     113395
lot2_surface_carrez             125324
lot3_numero                     128746
lot3_surface_carrez             130957
lot4_numero                     130716
lot4_surface_carrez      

In [28]:
(df_residentiel.isna().sum() / len(df_residentiel) * 100).round(1).sort_values(ascending=False)

ancien_nom_commune              100.0
ancien_code_commune             100.0
numero_volume                   100.0
ancien_id_parcelle              100.0
lot5_surface_carrez             100.0
lot4_surface_carrez              99.9
nature_culture_speciale          99.8
code_nature_culture_speciale     99.8
lot5_numero                      99.8
lot3_surface_carrez              99.6
lot4_numero                      99.5
lot3_numero                      98.0
lot2_surface_carrez              95.4
adresse_suffixe                  93.6
lot2_numero                      86.3
lot1_surface_carrez              79.6
lot1_numero                      63.8
surface_terrain                  37.1
nature_culture                   37.1
code_nature_culture              37.1
longitude                         2.4
latitude                          2.4
valeur_fonciere                   0.3
adresse_numero                    0.3
type_local                        0.0
surface_reelle_bati               0.0
code_type_lo

On élimine les valeurs nulles de la colonne "valeur foncière" qui sera très importante pour notre étude.

In [29]:
df_residentiel = df_residentiel.dropna(subset=["valeur_fonciere"])
print(df_residentiel.shape)

(131102, 41)


In [19]:
df_residentiel.isna().sum()

id_mutation                          0
date_mutation                        0
numero_disposition                   0
nature_mutation                      0
valeur_fonciere                      0
adresse_numero                     336
adresse_suffixe                 122754
adresse_nom_voie                     0
adresse_code_voie                    0
code_postal                          0
code_commune                         0
nom_commune                          0
code_departement                     0
ancien_code_commune             131102
ancien_nom_commune              131102
id_parcelle                          0
ancien_id_parcelle              131102
numero_volume                   131102
lot1_numero                      83614
lot1_surface_carrez             104253
lot2_numero                     113069
lot2_surface_carrez             124992
lot3_numero                     128414
lot3_surface_carrez             130624
lot4_numero                     130384
lot4_surface_carrez      

In [30]:
doublons_mutation = df_residentiel["id_mutation"].value_counts()
print(doublons_mutation.value_counts())

count
1      98976
2      11300
3       1092
4        486
5        135
6        115
8         53
7         43
9         24
10        21
12        11
11         8
15         6
14         5
18         4
13         3
16         3
21         3
17         2
22         2
19         1
105        1
101        1
26         1
100        1
92         1
76         1
75         1
72         1
68         1
56         1
52         1
46         1
40         1
38         1
34         1
32         1
30         1
28         1
20         1
Name: count, dtype: int64


Nous conservons uniquement les id_mutation qui apparaissent 1 seule fois. Si nous ne filtrons pas sur cet identifiant, nous risquons de compter plusieurs fois le même prix de vente (la valeur_fonciere est répétée sur chaque ligne d'une même mutation), ce qui aurait faussé les statistiques de prix. Nous gardons donc uniquement les id_mutation qui apparaissant une seule fois (ventes simples, un seul bien par transaction) pour simplifier l'analyse résidentielle. 

In [31]:
compte_par_mutation = df_residentiel.groupby("id_mutation").size()

# Garder uniquement les id_mutation qui apparaissent 1 seule fois
mutations_simples = compte_par_mutation[compte_par_mutation == 1].index
df_residentiel_clean = df_residentiel[df_residentiel["id_mutation"].isin(mutations_simples)].copy()

print(df_residentiel_clean.shape)

(98976, 41)


En excluant les mutations multi-lignes nous avons perdu ~11% de l'ensemble du dataset au départ (les ventes complexes:appartement + cave + parking en même temps)? Nous pourrons affiner notre analyse dans un second temps.

On observe maintenant les outliers:

In [32]:
df_residentiel_clean[["valeur_fonciere", "surface_reelle_bati"]].describe()

,valeur_fonciere,surface_reelle_bati
count,9.897600e+04,98974.000000
mean,2.625372e+05,79.635551
std,2.106826e+05,38.181039
min,1.000000e+00,2.000000
25%,1.500000e+05,53.000000
50%,2.200000e+05,75.000000
75%,3.200000e+05,99.000000
max,2.220500e+07,584.000000


Nous créons une colonne correspondant au prix au m2

In [33]:
df_residentiel_clean["prix_m2"] = df_residentiel_clean["valeur_fonciere"] / df_residentiel_clean["surface_reelle_bati"]
df_residentiel_clean["prix_m2"].describe()

count     98974.000000
mean       3461.851709
std        3068.987226
min           0.002564
25%        2436.363636
50%        3228.571429
75%        4103.756115
max      566250.000000
Name: prix_m2, dtype: float64

On observe nos valeurs aux extremums:

In [34]:
df_residentiel_clean.nsmallest(10, "prix_m2")[["valeur_fonciere", "surface_reelle_bati", "prix_m2", "type_local", "nom_commune"]]

,valeur_fonciere,surface_reelle_bati,prix_m2,type_local,nom_commune
168205,1.0,390.0,0.002564,Maison,Le Pouliguen
338568,1.0,161.0,0.006211,Maison,Saint-Malo-de-Guersac
283011,1.0,160.0,0.006250,Maison,Saint-Nazaire
374410,1.0,146.0,0.006849,Maison,Saffré
342323,1.0,145.0,0.006897,Maison,Guérande
259921,1.0,143.0,0.006993,Maison,Saint-Lyphard
192701,1.0,117.0,0.008547,Maison,Assérac
277260,1.0,113.0,0.008850,Maison,Saint-Lyphard
103947,1.0,100.0,0.010000,Maison,Saffré
195436,1.0,100.0,0.010000,Maison,Assérac


In [35]:
df_residentiel_clean.nlargest(10, "prix_m2")[["valeur_fonciere", "surface_reelle_bati", "prix_m2", "type_local", "nom_commune"]]

,valeur_fonciere,surface_reelle_bati,prix_m2,type_local,nom_commune
370467,1132500.0,2.0,566250.000000,Appartement,Nantes
231228,22205000.0,55.0,403727.272727,Maison,Geneston
287259,15947681.0,109.0,146309.000000,Maison,Orvault
383089,7719241.0,53.0,145646.056604,Appartement,Nantes
133981,5464757.5,40.0,136618.937500,Maison,Nantes
331607,4000000.0,30.0,133333.333333,Appartement,La Turballe
91927,250000.0,2.0,125000.000000,Appartement,La Baule-Escoublac
277175,2050000.0,18.0,113888.888889,Maison,La Baule-Escoublac
362463,210000.0,2.0,105000.000000,Appartement,Nantes
427072,4281680.0,43.0,99573.953488,Maison,Saint-Nazaire


On observe des valeurs abérrantes comme par exemplee un logement de 2m2 avec un prix au m2 de 566250€ ou encore des ventes de biens pour 1€ (cela correspond à des ventes faites pour 1€ symbolique de la part des propriétaire à leur famille par exemple).  
On établit donc un seuil pour trier nos données.Un prix au m2 situé entre 500 et 20000 € semble bon pour conserver des données cohérentes. Lorsque l'on observe les données pour des prix au m2 entre 15000 et 20000€ les lignes semblent correctes car il s'agit pour la plupart de logements en bord de mer et principalement à la Baule-Escoublac ou le côut des logements est élevé.

In [36]:
# Combien de lignes ont un prix_m2 entre 15000 et 20000 ?
zone_intermediaire = df_residentiel_clean[
    (df_residentiel_clean["prix_m2"] > 15000) &
    (df_residentiel_clean["prix_m2"] <= 20000)
]
print(zone_intermediaire.shape[0])
zone_intermediaire[["valeur_fonciere", "surface_reelle_bati", "prix_m2", "type_local", "nom_commune"]]

73


,valeur_fonciere,surface_reelle_bati,prix_m2,type_local,nom_commune
60440,1342056.0,80.0,16775.700000,Maison,Carquefou
67033,750000.0,49.0,15306.122449,Maison,Batz-sur-Mer
67592,1965000.0,110.0,17863.636364,Appartement,La Baule-Escoublac
73923,2150000.0,130.0,16538.461538,Appartement,La Baule-Escoublac
73963,1500000.0,80.0,18750.000000,Appartement,Pornic
...,...,...,...,...,...
418661,1190000.0,64.0,18593.750000,Appartement,La Baule-Escoublac
423546,1202000.0,78.0,15410.256410,Appartement,La Baule-Escoublac
425800,520000.0,34.0,15294.117647,Appartement,La Baule-Escoublac
427053,1000000.0,66.0,15151.515152,Appartement,La Baule-Escoublac


In [37]:
df_final = df_residentiel_clean[
    (df_residentiel_clean["prix_m2"] >= 500) &
    (df_residentiel_clean["prix_m2"] <= 20000)
].copy()

print(f"Avant filtrage : {df_residentiel_clean.shape[0]} lignes")
print(f"Après filtrage : {df_final.shape[0]} lignes")
print(f"Lignes supprimées : {df_residentiel_clean.shape[0] - df_final.shape[0]}")

Avant filtrage : 98976 lignes
Après filtrage : 97966 lignes
Lignes supprimées : 1010


Après élimination des valeurs abérrantes, 1010 lignes ont été supprimées. 

In [38]:
# Vérifier le format actuel de la colonne date
df_final["date_mutation"].head()
df_final["date_mutation"].dtype

dtype('O')

In [39]:
# Conversion en vrai format date
df_final["date_mutation"] = pd.to_datetime(df_final["date_mutation"])

print(df_final["date_mutation"].dtype)
df_final["date_mutation"].head()

datetime64[ns]


0   2021-01-05
2   2021-01-04
3   2021-01-05
5   2021-01-07
6   2021-01-04
Name: date_mutation, dtype: datetime64[ns]

In [40]:
df_final["annee"] = df_final["date_mutation"].dt.year
df_final[["date_mutation", "annee"]].head()

,date_mutation,annee
0,2021-01-05,2021
2,2021-01-04,2021
3,2021-01-05,2021
5,2021-01-07,2021
6,2021-01-04,2021


In [42]:
df_final["nombre_pieces_principales"].value_counts().sort_index()

nombre_pieces_principales
0.0        82
1.0      8327
2.0     18023
3.0     23743
4.0     25422
5.0     14288
6.0      5379
7.0      1781
8.0       601
9.0       205
10.0       62
11.0       24
12.0       17
13.0        6
14.0        1
16.0        1
19.0        3
34.0        1
Name: count, dtype: int64

In [58]:
df_final[df_final["nombre_pieces_principales"] >= 15][["valeur_fonciere", "surface_reelle_bati", "type_local", "nom_commune"]]

,valeur_fonciere,surface_reelle_bati,type_local,nom_commune
160682,500000.0,111.0,Maison,Thouaré-sur-Loire
170026,1610000.0,330.0,Maison,La Bernerie-en-Retz
176011,1925000.0,490.0,Maison,Guérande
316271,1000000.0,337.0,Maison,Clisson


Verification des valeurs aberrantes selon la surface des pièces:

In [60]:

df_final["surface_par_piece"] = df_final["surface_reelle_bati"] / df_final["nombre_pieces_principales"]


incoherents = df_final[
    (df_final["nombre_pieces_principales"] > 0) &
    (df_final["surface_par_piece"] <6)
]
print(incoherents.shape[0])
incoherents[["valeur_fonciere", "surface_reelle_bati", "nombre_pieces_principales", "surface_par_piece", "nom_commune"]]

4


,valeur_fonciere,surface_reelle_bati,nombre_pieces_principales,surface_par_piece,nom_commune
37456,50000.0,11.0,2.0,5.500000,Nantes
160682,500000.0,111.0,19.0,5.842105,Thouaré-sur-Loire
215933,3500.0,5.0,1.0,5.000000,Le Cellier
400391,150000.0,44.0,11.0,4.000000,La Haie-Fouassière


On constate des valeurs incohérentes: un logement de 72m2 qui comprte 34 pièces principales: ce n'est pas possible, nous allons donc la supprimer de notre dataset. idem pour le logement de 14m2 avec 5 pièces principales.

In [53]:
df_final = df_final[
    ~((df_final["nom_commune"] == "Orvault") & (df_final["nombre_pieces_principales"] == 34)) &
    ~((df_final["nom_commune"] == "Saint-Nazaire") & (df_final["surface_reelle_bati"] == 14) & (df_final["nombre_pieces_principales"] == 5))
].copy()

print(df_final.shape)

(97964, 45)


L'analyse porte sur les biens classés "Maison" 
ou "Appartement" dans les actes notariés. 
Cette catégorisation administrative peut inclure marginalement des biens 
atypiques (très petites surfaces historiques, biens à rénover...) 
qui ne correspondent pas à la définition légale d'un logement décent 
(≥9m², loi du 6 juillet 1989). Ces cas restent très minoritaires 
et n'impactent pas significativement les tendances observées, 
d'autant que les valeurs aberrantes de prix au m² ont été exclues 
lors du nettoyage.

Import du dataset Final Nettoyé:

In [61]:
df_final.to_csv(
    "df_final.csv",
    index=False,
    encoding="utf-8",
    sep=","
)